In [1]:
import re
import ast
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/processed/resume_data_clean.csv")
print(df.shape)

(9544, 43)


In [2]:
def normalize_skill(skill):
    return re.sub(r"\s+", " ", str(skill).strip().lower())

def skills_to_text(x):
    """Transforme la liste stringifiée 'skills' en texte espacé pour le TF-IDF."""
    try:
        items = ast.literal_eval(x)
        return " ".join(normalize_skill(s) for s in items if str(s).strip())
    except Exception:
        return ""

def required_to_text(x):
    if pd.isna(x):
        return ""
    return " ".join(normalize_skill(s) for s in x.split("\n") if s.strip())

df["resume_text"] = (
    df["career_objective_clean"].fillna("") + " " + df["skills"].apply(skills_to_text)
)
df["job_text"] = (
    df["responsibilities.1_clean"].fillna("") + " " +
    df["educationaL_requirements_clean"].fillna("") + " " +
    df["skills_required"].apply(required_to_text)
)

df[["resume_text", "job_text"]].sample(2, random_state=42)

,resume_text,job_text
5470,assembly autocad automation budgets budget cn...,data platform design data pipeline development...
8982,client engagement communication analytical an...,relationship building hr collaboration policy ...


In [3]:
df["experience_years_min"] = (
    df["experiencere_requirement"].str.extract(r"(\d+)").astype(float)
)
df["experience_years_min"] = df["experience_years_min"].fillna(0)

In [6]:
# Groupe = skills si disponible, sinon un identifiant unique par ligne
fallback_ids = pd.Series(
    "missing_resume_" + df.index.astype(str),
    index=df.index
)
group_ids = df["skills"].fillna(fallback_ids)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=group_ids))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

# Vérification : aucun CV en commun entre train et test (sur les skills non-nulles uniquement)
train_skills = set(df_train["skills"].dropna())
test_skills = set(df_test["skills"].dropna())
overlap = train_skills & test_skills
print(f"Train: {df_train.shape[0]} lignes | Test: {df_test.shape[0]} lignes | CV en commun: {len(overlap)}")

Train: 7519 lignes | Test: 2025 lignes | CV en commun: 0


In [7]:
tfidf = TfidfVectorizer(max_features=200, ngram_range=(1, 2), stop_words="english")

# Fit sur le vocabulaire du train (CV + poste combinés)
tfidf.fit(pd.concat([df_train["resume_text"], df_train["job_text"]]))

resume_tfidf_train = tfidf.transform(df_train["resume_text"])
job_tfidf_train = tfidf.transform(df_train["job_text"])
resume_tfidf_test = tfidf.transform(df_test["resume_text"])
job_tfidf_test = tfidf.transform(df_test["job_text"])

print(resume_tfidf_train.shape)  # (n_lignes_train, 200)

(7519, 200)


In [8]:
def row_cosine_similarity(mat_a, mat_b):
    """Similarité cosinus ligne par ligne entre deux matrices sparse de même forme."""
    sims = np.array([
        cosine_similarity(mat_a[i], mat_b[i])[0, 0] for i in range(mat_a.shape[0])
    ])
    return sims.reshape(-1, 1)

text_sim_train = row_cosine_similarity(resume_tfidf_train, job_tfidf_train)
text_sim_test = row_cosine_similarity(resume_tfidf_test, job_tfidf_test)

print("Similarité moyenne train:", text_sim_train.mean())

Similarité moyenne train: 0.05908540151837426


In [9]:
ohe = OneHotEncoder(handle_unknown="ignore")
ohe.fit(df_train[["job_position_name"]])

job_ohe_train = ohe.transform(df_train[["job_position_name"]])
job_ohe_test = ohe.transform(df_test[["job_position_name"]])

In [10]:
numeric_cols = [
    "skills_overlap", "experience_years_min",
    "has_career_objective", "has_certifications", "has_languages",
    "has_extra_curricular", "has_skills_required",
    "has_experience_requirement", "has_age_requirement",
]

X_train = sparse.hstack([
    resume_tfidf_train, job_tfidf_train,
    sparse.csr_matrix(text_sim_train),
    sparse.csr_matrix(df_train[numeric_cols].values),
    job_ohe_train,
]).tocsr()

X_test = sparse.hstack([
    resume_tfidf_test, job_tfidf_test,
    sparse.csr_matrix(text_sim_test),
    sparse.csr_matrix(df_test[numeric_cols].values),
    job_ohe_test,
]).tocsr()

y_train = df_train["matched_score"].values
y_test = df_test["matched_score"].values

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (7519, 438) | X_test: (2025, 438)


In [11]:
import joblib

sparse.save_npz("../data/processed/X_train.npz", X_train)
sparse.save_npz("../data/processed/X_test.npz", X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy", y_test)

joblib.dump(tfidf, "../data/processed/tfidf_vectorizer.pkl")
joblib.dump(ohe, "../data/processed/job_position_encoder.pkl")

print("Features et encodeurs sauvegardés.")

Features et encodeurs sauvegardés.
